<a href="https://colab.research.google.com/github/Aham0803/PySpark/blob/main/DAG%20AND%20LAZY.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
! pip install pyspark

imported the required libraries

In [3]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import avg , col , lit, sum , count , desc,when,row_number
from pyspark.sql.window import Window

creating a session

In [4]:
spark = SparkSession.builder.appName("PySpark_Basics").getOrCreate()
print(f"spark session created successfullly with spark version :{spark.version}")

spark session created successfullly with spark version :4.0.3


create a random data

In [5]:
employee_data =[
    (101,"Gaurav yadav" , "IT" , 100000 ,"New Delhi"),
    (102,"Dev yadav" , "IT" , 80000 ,"Mumbai"),
    (103,"Gaurav kumar" , "Sales" , 70000 ,"Chennai"),
    (104,"Amit yadav" , "IT" , 100000 ,"New Delhi"),
    (105,"Divya yadav" , "HR" , 50000 ,"Banglore"),
    (106,"Ajay yadav" , "IT" , 90000 ,"Ahemdabad"),
    (106,"Gaurav Raj" , "IT" , 10000 ,"Pune"),
]
column = ["emp_id" , "emp_name" , "department" , "salary" ,"city"]
df = spark.createDataFrame(
    data = employee_data,
    schema = column
)

In [6]:
print("original schema\n")
df.show()
df.printSchema()

original schema

+------+------------+----------+------+---------+
|emp_id|    emp_name|department|salary|     city|
+------+------------+----------+------+---------+
|   101|Gaurav yadav|        IT|100000|New Delhi|
|   102|   Dev yadav|        IT| 80000|   Mumbai|
|   103|Gaurav kumar|     Sales| 70000|  Chennai|
|   104|  Amit yadav|        IT|100000|New Delhi|
|   105| Divya yadav|        HR| 50000| Banglore|
|   106|  Ajay yadav|        IT| 90000|Ahemdabad|
|   106|  Gaurav Raj|        IT| 10000|     Pune|
+------+------------+----------+------+---------+

root
 |-- emp_id: long (nullable = true)
 |-- emp_name: string (nullable = true)
 |-- department: string (nullable = true)
 |-- salary: long (nullable = true)
 |-- city: string (nullable = true)



Data Transformation

1 . Selecting and renaming columns

In [7]:
df_selected = df.select(col('emp_id') ,col('emp_name').alias ("full_name") , col('salary'))
df_selected.show(5)

+------+------------+------+
|emp_id|   full_name|salary|
+------+------------+------+
|   101|Gaurav yadav|100000|
|   102|   Dev yadav| 80000|
|   103|Gaurav kumar| 70000|
|   104|  Amit yadav|100000|
|   105| Divya yadav| 50000|
+------+------------+------+
only showing top 5 rows


2. filtering rows

In [8]:
df_filtered = df.filter((col('department') == "IT") & (col('salary') >80000))
df_filtered.show()

+------+------------+----------+------+---------+
|emp_id|    emp_name|department|salary|     city|
+------+------------+----------+------+---------+
|   101|Gaurav yadav|        IT|100000|New Delhi|
|   104|  Amit yadav|        IT|100000|New Delhi|
|   106|  Ajay yadav|        IT| 90000|Ahemdabad|
+------+------------+----------+------+---------+



3. adding and modifying columns

In [9]:
df_transformed = df.withColumn("bonus" , col ("salary") * 0.10).withColumn("salary_tier" , when(col("salary")>= 90000 , lit("High")).otherwise(lit("standard")))
df_transformed.show()

+------+------------+----------+------+---------+-------+-----------+
|emp_id|    emp_name|department|salary|     city|  bonus|salary_tier|
+------+------------+----------+------+---------+-------+-----------+
|   101|Gaurav yadav|        IT|100000|New Delhi|10000.0|       High|
|   102|   Dev yadav|        IT| 80000|   Mumbai| 8000.0|   standard|
|   103|Gaurav kumar|     Sales| 70000|  Chennai| 7000.0|   standard|
|   104|  Amit yadav|        IT|100000|New Delhi|10000.0|       High|
|   105| Divya yadav|        HR| 50000| Banglore| 5000.0|   standard|
|   106|  Ajay yadav|        IT| 90000|Ahemdabad| 9000.0|       High|
|   106|  Gaurav Raj|        IT| 10000|     Pune| 1000.0|   standard|
+------+------------+----------+------+---------+-------+-----------+



In [10]:
df_transformed.explain(True)

== Parsed Logical Plan ==
'Project [unresolvedstarwithcolumns(salary_tier, CASE WHEN '`>=`('salary, 90000) THEN High ELSE standard END, None)]
+- Project [emp_id#0L, emp_name#1, department#2, salary#3L, city#4, (cast(salary#3L as double) * 0.1) AS bonus#48]
   +- LogicalRDD [emp_id#0L, emp_name#1, department#2, salary#3L, city#4], false

== Analyzed Logical Plan ==
emp_id: bigint, emp_name: string, department: string, salary: bigint, city: string, bonus: double, salary_tier: string
Project [emp_id#0L, emp_name#1, department#2, salary#3L, city#4, bonus#48, CASE WHEN (salary#3L >= cast(90000 as bigint)) THEN High ELSE standard END AS salary_tier#49]
+- Project [emp_id#0L, emp_name#1, department#2, salary#3L, city#4, (cast(salary#3L as double) * 0.1) AS bonus#48]
   +- LogicalRDD [emp_id#0L, emp_name#1, department#2, salary#3L, city#4], false

== Optimized Logical Plan ==
Project [emp_id#0L, emp_name#1, department#2, salary#3L, city#4, (cast(salary#3L as double) * 0.1) AS bonus#48, CASE W

4. Aggregation and grouping

In [11]:
df_agg = df.groupby("department").agg(avg("salary").alias("avg_salary"),count("emp_id").alias("emp_count")).orderBy(desc("avg_salary"))
df_agg.show()

+----------+----------+---------+
|department|avg_salary|emp_count|
+----------+----------+---------+
|        IT|   76000.0|        5|
|     Sales|   70000.0|        1|
|        HR|   50000.0|        1|
+----------+----------+---------+



5. window function

In [12]:
window_spec = Window.partitionBy("department").orderBy(desc("salary"))
df_ranked = df.withColumn("salary_rank" , row_number().over(window_spec))
df_ranked.show()

+------+------------+----------+------+---------+-----------+
|emp_id|    emp_name|department|salary|     city|salary_rank|
+------+------------+----------+------+---------+-----------+
|   105| Divya yadav|        HR| 50000| Banglore|          1|
|   101|Gaurav yadav|        IT|100000|New Delhi|          1|
|   104|  Amit yadav|        IT|100000|New Delhi|          2|
|   106|  Ajay yadav|        IT| 90000|Ahemdabad|          3|
|   102|   Dev yadav|        IT| 80000|   Mumbai|          4|
|   106|  Gaurav Raj|        IT| 10000|     Pune|          5|
|   103|Gaurav kumar|     Sales| 70000|  Chennai|          1|
+------+------------+----------+------+---------+-----------+



stoppping the spark session

In [13]:
spark.stop()
print("\n Spark Session Stopped successfullly")


 Spark Session Stopped successfullly


## Directed Acylic Graph(DAG)

1. Initialization and setup

In [14]:
import time
from pyspark.sql import SparkSession
from pyspark.sql.functions import col , lit


Session Creation

In [15]:
spark = SparkSession.builder.appName("DAG Session").getOrCreate()
print("Spark Session has been created succeffullly")

Spark Session has been created succeffullly


Lazy Evaluation

## created a large range of numbers(10 million)

In [16]:
start_time = time.time()
df_range = spark.range(0,10000000)
print(f"time to define range: {time.time()- start_time: .4f} seconds(Almost Instantaneous)")

time to define range:  0.0545 seconds(Almost Instantaneous)


### ADD an expensive mathematical transformation

In [17]:
df_transformed = df_range.withColumn("multiplied_val" , col("id")*3).filter(col("multiplied_val") % 2 == 0)
print(f"time to define transformation :{time.time() - start_time: .4f} seconds(Still no work is done)")

time to define transformation : 0.1693 seconds(Still no work is done)


## viewing the logical and physical DAG

In [18]:
df_transformed.explain(True)

== Parsed Logical Plan ==
'Filter '`=`('`%`('multiplied_val, 2), 0)
+- Project [id#121L, (id#121L * cast(3 as bigint)) AS multiplied_val#122L]
   +- Range (0, 10000000, step=1, splits=Some(2))

== Analyzed Logical Plan ==
id: bigint, multiplied_val: bigint
Filter ((multiplied_val#122L % cast(2 as bigint)) = cast(0 as bigint))
+- Project [id#121L, (id#121L * cast(3 as bigint)) AS multiplied_val#122L]
   +- Range (0, 10000000, step=1, splits=Some(2))

== Optimized Logical Plan ==
Project [id#121L, (id#121L * 3) AS multiplied_val#122L]
+- Filter (((id#121L * 3) % 2) = 0)
   +- Range (0, 10000000, step=1, splits=Some(2))

== Physical Plan ==
*(1) Project [id#121L, (id#121L * 3) AS multiplied_val#122L]
+- *(1) Filter (((id#121L * 3) % 2) = 0)
   +- *(1) Range (0, 10000000, step=1, splits=2)



## Trigger the action

In [22]:
action_start_time = time.time()
total_rows = df_transformed.count()
action_end_time = time.time()

print(f"Resulting Row Count :{total_rows}")
print(f"Time taken to execute action :{action_end_time - action_start_time:.2f} seconds")

Resulting Row Count :5000000
Time taken to execute action :0.70 seconds


## Performance Optimization

In [25]:
# Rerun the whole action without cache
start_time = time.time()
df_transformed.count()
print(f"This for 2nd run(no cache):{time.time()- start_time: .2f}seconds")

This for 2nd run(no cache): 0.26seconds


In [27]:
# Rerun with cache
# .cache() is a lazy transformation that tells spark to store the result in memoery once computed
df_transformed.cache()
df_transformed.count()
start_time = time.time()
df_transformed.count()
print(f"Time for 3rd run (Cached) : {time.time() - start_time: .2f} seconds")

Time for 3rd run (Cached) :  0.26 seconds
